# Flower Classification - Model Evaluation
## Comprehensive Evaluation and Visualization

This notebook evaluates the trained models with detailed metrics and visualizations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Load Data and Models

In [ ]:
# Load test data
data_dir = Path('../data/processed')
X_test = np.load(data_dir / 'X_test.npy')
y_test = np.load(data_dir / 'y_test.npy')

# Load label encoder
models_dir = Path('../models/artifacts')
label_encoder = joblib.load(models_dir / 'label_encoder.joblib')
class_names = label_encoder.classes_

# Load all trained models
models = {}
model_files = ['logistic_regression_model.joblib', 'decision_tree_model.joblib',
               'random_forest_model.joblib', 'xgboost_model.joblib']

for model_file in model_files:
    model_path = models_dir / model_file
    if model_path.exists():
        model_name = model_file.replace('_model.joblib', '').replace('_', ' ').title()
        models[model_name] = joblib.load(model_path)
        print(f'Loaded {model_name}')

print(f'\nLoaded {len(models)} models')
print(f'Classes: {class_names}')
print(f'Test samples: {len(X_test)}')

## 2. Generate Predictions

In [ ]:
# Generate predictions for all models
predictions = {}

for name, model in models.items():
    predictions[name] = model.predict(X_test)
    print(f'{name}: Predictions generated')

print(f'\nAll predictions generated for {len(predictions)} models')

## 3. Compute Metrics for All Models

In [ ]:
# Compute all metrics for each model
all_metrics = {}

for name, y_pred in predictions.items():
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0)
    }
    all_metrics[name] = metrics
    
    print(f'\n{name}:')
    print(f'  Accuracy:  {metrics["accuracy"]:.4f}')
    print(f'  Precision: {metrics["precision"]:.4f}')
    print(f'  Recall:    {metrics["recall"]:.4f}')
    print(f'  F1 Score:  {metrics["f1"]:.4f}')

# Create metrics dataframe
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.sort_values('accuracy', ascending=False)
print('\n' + '='*50)
print('Model Comparison:')
print('='*50)
metrics_df

## 4. Visualize Metrics Comparison

In [ ]:
# Plot metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

metrics_names = ['accuracy', 'precision', 'recall', 'f1']
titles = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
colors = sns.color_palette('viridis', len(models))

for idx, (metric, title) in enumerate(zip(metrics_names, titles)):
    ax = axes[idx]
    values = [all_metrics[model][metric] for model in all_metrics]
    bars = ax.bar(range(len(models)), values, color=colors)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(all_metrics.keys(), rotation=45, ha='right')
    ax.set_ylabel(title)
    ax.set_title(f'{title} Comparison')
    ax.set_ylim(0, 1.05)
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Confusion Matrix

In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[idx]
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(f'{name}\nAccuracy: {all_metrics[name]["accuracy"]:.4f}')

plt.tight_layout()
plt.show()

## 6. Detailed Classification Report (Best Model)

In [ ]:
# Get best model
best_model_name = metrics_df.index[0]
y_pred_best = predictions[best_model_name]

print(f'Classification Report - {best_model_name}')
print('='*60)
report = classification_report(y_test, y_pred_best, target_names=class_names, zero_division=0)
print(report)

## 7. Analyze Misclassifications

In [ ]:
# Analyze misclassified samples for best model
misclassified_mask = y_test != y_pred_best
misclassified_indices = np.where(misclassified_mask)[0]

print(f'{best_model_name} - Misclassification Analysis')
print('='*60)
print(f'Total misclassified: {len(misclassified_indices)} / {len(y_test)} ({len(misclassified_indices)/len(y_test)*100:.1f}%)')

# Confusion pattern analysis
cm = confusion_matrix(y_test, y_pred_best)
print(f'\nConfusion Pattern:')
for i, true_class in enumerate(class_names):
    for j, pred_class in enumerate(class_names):
        if i != j and cm[i, j] > 0:
            print(f'  {true_class} → {pred_class}: {cm[i, j]} sample(s)')

In [ ]:
# Show misclassified examples
if len(misclassified_indices) > 0:
    print(f'\nMisclassified Examples:')
    print('-'*60)
    for idx in misclassified_indices[:5]:  # Show first 5
        true_label = class_names[y_test[idx]]
        pred_label = class_names[y_pred_best[idx]]
        print(f'Sample {idx}: True={true_label}, Predicted={pred_label}')

## 8. Model Ranking Summary

In [ ]:
# Create ranking visualization
plt.figure(figsize=(12, 6))

# Calculate composite score (average of all metrics)
metrics_df['composite'] = metrics_df[metrics_names].mean(axis=1)
metrics_df = metrics_df.sort_values('composite', ascending=False)

# Plot
bars = plt.bar(range(len(metrics_df)), metrics_df['composite'], 
               color=sns.color_palette('viridis', len(metrics_df)))
plt.xticks(range(len(metrics_df)), metrics_df.index, rotation=45, ha='right')
plt.ylabel('Composite Score (Avg of all metrics)')
plt.title('Model Ranking - Composite Score')
plt.ylim(0, 1.05)

for bar, val in zip(bars, metrics_df['composite']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Print ranking
print('\nFinal Model Ranking:')
print('='*60)
for rank, (model_name, row) in enumerate(metrics_df.iterrows(), 1):
    print(f'{rank}. {model_name}: {row["composite"]:.4f} (Accuracy: {row["accuracy"]:.4f})')

## 9. Save Evaluation Results

In [ ]:
# Save evaluation results
results_dir = Path('../models/artifacts')
results_dir.mkdir(parents=True, exist_ok=True)

# Save metrics
metrics_df.to_csv(results_dir / 'evaluation_metrics.csv')
print(f'Saved evaluation metrics to evaluation_metrics.csv')

# Save confusion matrices
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    np.save(results_dir / f'{name.lower().replace(" ", "_")}_confusion_matrix.npy', cm)
print(f'Saved confusion matrices')

# Save best model info
best_model_info = {
    'model_name': best_model_name,
    'metrics': all_metrics[best_model_name],
    'class_names': list(class_names)
}
joblib.dump(best_model_info, results_dir / 'best_model_info.joblib')
print(f'Saved best model info: {best_model_name}')

## Summary

✅ Evaluated **{} models** on test set  
✅ Computed **accuracy, precision, recall, F1** for all models  
✅ Generated **confusion matrices** and **classification reports**  
✅ Analyzed **misclassification patterns**  
✅ Created **model ranking** based on composite score  
✅ **Best model**: **{best_model_name}** with **{all_metrics[best_model_name]["accuracy"]:.4f}** accuracy  
✅ Saved all evaluation results and artifacts